In [36]:
import numpy as np
from helpers import load_csv_data
from collections import defaultdict

In [8]:
#Load data
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("dataset")

## 1. Inspect raw data

In [9]:
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_train unique values:", np.unique(y_train))

# dtype
print("x_train dtype:", x_train.dtype)

# NaN count per column (if NaNs are already np.nan)
nan_counts = np.isnan(x_train).sum(axis=0)
print("Columns with NaNs:", np.sum(nan_counts > 0), "out of", x_train.shape[1])
print("Max NaN fraction in a column:", nan_counts.max() / x_train.shape[0])

# Basic stats per column
print("Min per column (first 10):", np.nanmin(x_train[:, :10], axis=0))
print("Max per column (first 10):", np.nanmax(x_train[:, :10], axis=0))

x_train shape: (328135, 321)
x_test shape: (109379, 321)
y_train shape: (328135,)
y_train unique values: [-1  1]
x_train dtype: float64
Columns with NaNs: 239 out of 321
Max NaN fraction in a column: 0.9999024791625398
Min per column (first 10): [1.000000e+00 1.000000e+00 1.012016e+06 1.000000e+00 1.000000e+00
 2.015000e+03 1.100000e+03 2.015000e+09 2.015000e+09 1.000000e+00]
Max per column (first 10): [7.20000000e+01 1.20000000e+01 1.23120150e+07 1.20000000e+01
 3.10000000e+01 2.01600000e+03 1.20000000e+03 2.01502324e+09
 2.01502324e+09 1.00000000e+00]


## 2. Data cleaning
### 2.1 Distribution of NaN fractions across columns

In [10]:
nan_frac = np.isnan(x_train).sum(axis=0) / x_train.shape[0]

# Histogram-style summary of how "bad" columns are
print("Columns with 0% NaN:", np.sum(nan_frac == 0))
print("Columns with <5% NaN:", np.sum((nan_frac > 0) & (nan_frac < 0.05)))
print("Columns with 5-50% NaN:", np.sum((nan_frac >= 0.05) & (nan_frac < 0.5))) 
print("Columns with 50-90% NaN:", np.sum((nan_frac >= 0.5) & (nan_frac < 0.9)))
print("Columns with >90% NaN:", np.sum(nan_frac >= 0.9))

Columns with 0% NaN: 82
Columns with <5% NaN: 33
Columns with 5-50% NaN: 59
Columns with 50-90% NaN: 48
Columns with >90% NaN: 99


### 2.2 Drop non-predictive administrative columns

In [21]:
with open("dataset/x_train.csv", "r") as f:
    header = f.readline().strip()
feature_names = header.split(",")
feature_names = feature_names[1:]  # drop 'Id'
print(len(feature_names))
print(feature_names[:5])

321
['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY']


In [22]:
admin_cols = ['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 
              'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'COLGHOUS',
              'STATERES', 'CELLFON3', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN',
              'CTELNUM1', 'CELLFON2', 'CADULT', 'PVTRESD2', 'CCLGHOUS', 'CSTATE',
              'LANDLINE', 'HHADULT', 'QSTVER', 'QSTLANG', 'MSCODE',
              '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_CLLCPWT', '_DUALUSE',
              '_DUALCOR', '_LLCPWT']


admin_idx = [feature_names.index(c) for c in admin_cols if c in feature_names]
print("Dropping", len(admin_idx), "admin columns")

keep_mask = np.ones(len(feature_names), dtype=bool)
keep_mask[admin_idx] = False

x_train_clean = x_train[:, keep_mask]
x_test_clean = x_test[:, keep_mask]
feature_names_clean = [f for f, k in zip(feature_names, keep_mask) if k]

print(x_train_clean.shape)
print(x_test_clean.shape)
print(len(feature_names_clean))

Dropping 37 admin columns
(328135, 284)
(109379, 284)
284


### 2.3 Convert BRFSS "missing" sentinel codes to real NaN

This is the critical BRFSS-specific step. In this survey, missingness is usually hidden as specific numeric codes rather than true NaN:

Categorical variables: 7 = "Don't know/Not sure", 9 = "Refused"
Two-digit variables: 77 = "Don't know", 99 = "Refused"
Continuous variables (like PHYSHLTH, MENTHLTH): 77/88/99 or 777/999/7777/9999 depending on scale
77/99 at boundary of the variable's natural range is the giveaway — e.g. if a column's real values are 1–30 but it also has 77 and 99, those are sentinel codes, not real data points.

Rather than hardcoding all ~230 variable-specific rules (the official codebook has each one individually), here's a practical heuristic that works well for BRFSS: for each column, if the max value is one of {7, 9, 77, 88, 99, 777, 999, 7777, 9999} and it sits far above the bulk of the other values (i.e., it's a clear outlier code, not a natural continuation of the scale), treat it as missing.

In [24]:
sentinel_codes = {7, 9, 77, 88, 99, 777, 999, 7777, 9999}

def replace_sentinels(x, col_idx, verbose=False):
    col = x[:, col_idx].copy()
    non_nan = col[~np.isnan(col)]
    if len(non_nan) == 0:
        return col
    for code in sentinel_codes:
        if code in non_nan:
            # only treat as sentinel if it's clearly separated from the rest
            rest = non_nan[non_nan != code]
            if len(rest) > 0 and code > np.percentile(rest, 99) + 1:
                col[col == code] = np.nan
                if verbose:
                    print(f"Column {col_idx}: replaced code {code}")
    return col

x_train_sentinel = x_train_clean.copy()
for i in range(x_train_sentinel.shape[1]):
    x_train_sentinel[:, i] = replace_sentinels(x_train_sentinel, i)

# Check the impact on train:
before = np.isnan(x_train_clean).sum()
after = np.isnan(x_train_sentinel).sum()
print("NaNs before:", before)
print("NaNs after:", after)
print("Newly identified missing values:", after - before)


NaNs before: 43341504
NaNs after: 44757010
Newly identified missing values: 1415506


In [25]:
# Apply the identical replacement to test.
def get_sentinel_codes_for_col(x_train, col_idx):
    col = x_train[:, col_idx]
    non_nan = col[~np.isnan(col)]
    found = set()
    if len(non_nan) == 0:
        return found
    for code in sentinel_codes:
        if code in non_nan:
            rest = non_nan[non_nan != code]
            if len(rest) > 0 and code > np.percentile(rest, 99) + 1:
                found.add(code)
    return found

# Determine sentinel codes using x_train only
col_sentinels = [get_sentinel_codes_for_col(x_train_clean, i) for i in range(x_train_clean.shape[1])]

# Apply the same codes to test
x_test_sentinel = x_test_clean.copy()
for i, codes in enumerate(col_sentinels):
    for code in codes:
        x_test_sentinel[x_test_clean[:, i] == code, i] = np.nan

print("Test NaNs before:", np.isnan(x_test_clean).sum())
print("Test NaNs after:", np.isnan(x_test_sentinel).sum())

Test NaNs before: 14446569
Test NaNs after: 14919295


### 2.4 Drop columns with excessive missingness

In [26]:
#recompute missingness fractions on the updated arrays
nan_frac = np.isnan(x_train_sentinel).sum(axis=0) / x_train_sentinel.shape[0]

# Recheck distribution
print("0% NaN:", np.sum(nan_frac == 0))
print("<5% NaN:", np.sum((nan_frac > 0) & (nan_frac < 0.05)))
print("5-50% NaN:", np.sum((nan_frac >= 0.05) & (nan_frac < 0.5)))
print("50-90% NaN:", np.sum((nan_frac >= 0.5) & (nan_frac < 0.9)))
print(">90% NaN:", np.sum(nan_frac >= 0.9))

0% NaN: 17
<5% NaN: 60
5-50% NaN: 70
50-90% NaN: 41
>90% NaN: 96


In [27]:
# look at which columns have >90% missing values:
high_missing_idx = np.where(nan_frac >= 0.9)[0]
high_missing_names = [feature_names_clean[i] for i in high_missing_idx]
print(len(high_missing_names))
print(high_missing_names)

96
['NUMPHON2', 'INSULIN', 'BLDSUGAR', 'FEETCHK2', 'DOCTDIAB', 'CHKHEMO3', 'FEETCHK', 'EYEEXAM', 'DIABEYE', 'DIABEDU', 'CRGVREL1', 'CRGVLNG1', 'CRGVHRS1', 'CRGVPRB1', 'CRGVPERS', 'CRGVHOUS', 'CRGVMST2', 'VIDFCLT2', 'VIREDIF3', 'VIPRFVS2', 'VINOCRE2', 'VIEYEXM2', 'VIINSUR2', 'VICTRCT4', 'VIGLUMA2', 'VIMACDG2', 'CDHOUSE', 'CDASSIST', 'CDHELP', 'CDSOCIAL', 'CDDISCUS', 'WTCHSALT', 'LONGWTCH', 'DRADVISE', 'ASTHMAGE', 'ASATTACK', 'ASERVIST', 'ASDRVIST', 'ASRCHKUP', 'ASACTLIM', 'ASYMPTOM', 'ASNOSLEP', 'ASTHMED3', 'ASINHALR', 'HAREHAB1', 'STREHAB1', 'CVDASPRN', 'ASPUNSAF', 'RLIVPAIN', 'RDUCHART', 'RDUCSTRK', 'ARTTODAY', 'ARTHWGT', 'ARTHEXER', 'ARTHEDU', 'TETANUS', 'HPVADVC2', 'HPVADSHT', 'SHINGLE2', 'HADMAM', 'HOWLONG', 'HADPAP2', 'LASTPAP2', 'HPVTEST', 'HPLSTTST', 'HADHYST2', 'PROFEXAM', 'LENGEXAM', 'LSTBLDS3', 'HADSGCO1', 'LASTSIG3', 'PCPSAAD2', 'PCPSADI1', 'PCPSARE1', 'PSATEST1', 'PSATIME', 'PCPSARS1', 'PCPSADE1', 'PCDMDECN', 'SCNTPAID', 'SCNTWRK1', 'SCNTLPAD', 'SCNTLWK1', 'CASTHNO2', 'EMTS

In [28]:
# Drop all 96 high-missing columns (None of these are core cardiovascular risk factors, they're mostly downstream management/detail questions for people who already have a specific condition.): 
high_missing_idx_set = set(high_missing_idx)  # from before, based on nan_frac >= 0.9

keep_mask2 = np.ones(x_train_sentinel.shape[1], dtype=bool)
keep_mask2[list(high_missing_idx_set)] = False

x_train_v2 = x_train_sentinel[:, keep_mask2]
x_test_v2 = x_test_sentinel[:, keep_mask2]
feature_names_v2 = [f for f, k in zip(feature_names_clean, keep_mask2) if k]

print(x_train_v2.shape)
print(x_test_v2.shape)
print(len(feature_names_v2))

(328135, 188)
(109379, 188)
188


### 2.5 Impute remaining missing values:

In [29]:
nan_frac_v2 = np.isnan(x_train_v2).sum(axis=0) / x_train_v2.shape[0]
print("Still have NaNs in", np.sum(nan_frac_v2 > 0), "out of", x_train_v2.shape[1], "columns")
print("Max remaining missing fraction:", nan_frac_v2.max())

Still have NaNs in 171 out of 188 columns
Max remaining missing fraction: 0.8853002575159614


In [30]:
#Look at remaining data with >50% missing values:
mid_missing_idx = np.where((nan_frac_v2 >= 0.5) & (nan_frac_v2 < 0.9))[0]
mid_missing_names = [feature_names_v2[i] for i in mid_missing_idx]
for name, frac in sorted(zip(mid_missing_names, nan_frac_v2[mid_missing_idx]), key=lambda x: -x[1]):
    print(f"{name:15s} {frac:.4f}")

CASTHDX2        0.8853
BLDSTOOL        0.8785
HADSIGM3        0.8781
DIABAGE2        0.8713
RCSGENDR        0.8707
ASTHNOW         0.8701
RCSRLTN2        0.8697
STOPSMK2        0.8614
_CHISPNC        0.8593
_CRACE1         0.8590
_CPRACE         0.8590
PREGNANT        0.8534
SCNTMNY1        0.8427
SCNTMEL1        0.8345
CRGVEXPT        0.8226
PDIABTST        0.8214
PREDIAB1        0.8134
CAREGIV1        0.7544
WHRTST10        0.7473
HIVTSTD3        0.7418
CIMEMLOS        0.7383
LASTSMK2        0.7253
JOINPAIN        0.7040
_PNEUMO2        0.6966
ARTHDIS2        0.6955
LMTJOIN3        0.6952
ARTHSOCL        0.6951
_FLSHOT6        0.6868
SXORIENT        0.6281
TRNSGNDR        0.6272
BPMEDS          0.5990
SMOKDAY2        0.5840
FLSHTMY2        0.5692
IMFVPLAC        0.5684
PADUR2_         0.5627
PAFREQ2_        0.5575
EXEROFT2        0.5574
EXERHMM2        0.5538
MAXDRNKS        0.5375
AVEDRNK2        0.5288
DRNK3GE5        0.5232


In [31]:
# Drop all features with > 50% missing values. Might be worth adding BPMEDS, AVEDRNK2, MAXDRNKS, DRNK3GE5 later to improve model if necessary
mid_missing_idx_set = set(mid_missing_idx)  # from before

keep_mask3 = np.ones(x_train_v2.shape[1], dtype=bool)
keep_mask3[list(mid_missing_idx_set)] = False

x_train_v3 = x_train_v2[:, keep_mask3]
x_test_v3 = x_test_v2[:, keep_mask3]
feature_names_v3 = [f for f, k in zip(feature_names_v2, keep_mask3) if k]

print(x_train_v3.shape)
print(x_test_v3.shape)
print(len(feature_names_v3))

(328135, 147)
(109379, 147)
147


In [33]:
# impute median if Nan value:
## worth trying other imputation techniques later on to improve outcome
nan_frac_v3 = np.isnan(x_train_v3).sum(axis=0) / x_train_v3.shape[0]
print("Columns with NaNs:", np.sum(nan_frac_v3 > 0), "out of", x_train_v3.shape[1])
print("Max remaining missing fraction:", nan_frac_v3.max())

col_medians = np.nanmedian(x_train_v3, axis=0)
print("Columns with NaN median (fully empty):", np.sum(np.isnan(col_medians)))

def impute_with_medians(x, medians):
    x_out = x.copy()
    inds = np.where(np.isnan(x_out))
    x_out[inds] = np.take(medians, inds[1])
    return x_out

x_train_imputed = impute_with_medians(x_train_v3, col_medians)
x_test_imputed = impute_with_medians(x_test_v3, col_medians)

print("Remaining NaNs in train:", np.isnan(x_train_imputed).sum())
print("Remaining NaNs in test:", np.isnan(x_test_imputed).sum())

Columns with NaNs: 130 out of 147
Max remaining missing fraction: 0.4898806893504198
Columns with NaN median (fully empty): 0
Remaining NaNs in train: 0
Remaining NaNs in test: 0


### 2.6 Remove near-constant (zero/low-variance) columns:

A column where almost every respondent has the same value carries little to no predictive power, but still adds noise and computation.

In [34]:
col_std = np.std(x_train_imputed, axis=0)
print("Columns with std == 0 (constant):", np.sum(col_std == 0))
print("Columns with very low std (< 0.01):", np.sum(col_std < 0.01))

# Look at the lowest-variance columns
low_var_idx = np.argsort(col_std)[:15]
for i in low_var_idx:
    print(f"{feature_names_v3[i]:15s} std={col_std[i]:.6f}")

Columns with std == 0 (constant): 0
Columns with very low std (< 0.01): 0
_VEG23          std=0.010901
_FRT16          std=0.014604
HTM4            std=0.103858
NUMHHOL2        std=0.170973
CHCKIDNY        std=0.183797
CVDSTRK3        std=0.197672
DIFFDRES        std=0.201203
BLIND           std=0.212923
PAMISS1_        std=0.214419
_RFSEAT2        std=0.215781
_RFDRHV5        std=0.218123
_HCVU651        std=0.250096
HLTHPLN1        std=0.259164
DIFFALON        std=0.263813
CHCCOPD1        std=0.270987


None of these should be dropped: low variance due to genuine class imbalance (rare condition) is very different from low variance due to a column being uninformative/constant.

### 2.7 Check correlation btw. features

In [35]:
# Compute correlation matrix (147x147)
corr_matrix = np.corrcoef(x_train_imputed, rowvar=False)

# Find pairs with |correlation| > 0.9, excluding the diagonal
high_corr_pairs = []
n = corr_matrix.shape[0]
for i in range(n):
    for j in range(i+1, n):
        if abs(corr_matrix[i, j]) > 0.9:
            high_corr_pairs.append((feature_names_v3[i], feature_names_v3[j], corr_matrix[i, j]))

print(f"Found {len(high_corr_pairs)} highly correlated pairs")
for a, b, c in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
    print(f"{a:15s} <-> {b:15s}  corr={c:.3f}")

Found 40 highly correlated pairs
TOLDHI2         <-> _RFCHOL          corr=-1.000
HAVARTH3        <-> _DRDXAR1         corr=1.000
EXERANY2        <-> _TOTINDA         corr=1.000
HIVTST6         <-> _AIDTST3         corr=1.000
ASTHMA3         <-> _LTASTH1         corr=-1.000
MAXVO2_         <-> FC60_            corr=1.000
_VEGRESP        <-> _VEGETEX         corr=-0.998
_PRACE1         <-> _MRACE1          corr=0.995
_FRTRESP        <-> _FRUITEX         corr=-0.995
HTIN4           <-> HTM4             corr=0.994
EXERHMM1        <-> PADUR1_          corr=0.993
BPHIGH4         <-> _RFHYPE5         corr=-0.991
EDUCA           <-> _EDUCAG          corr=0.988
_AGEG5YR        <-> _AGE80           corr=0.977
_AGE80          <-> _AGE_G           corr=0.971
_AGE80          <-> FC60_            corr=-0.969
_AGE80          <-> MAXVO2_          corr=-0.969
_MISFRTN        <-> _FRTRESP         corr=-0.964
BLOODCHO        <-> _CHOLCHK         corr=0.963
_MISFRTN        <-> _FRUITEX         corr=0.960

In [37]:
# Build a graph of highly correlated features and find connected groups

graph = defaultdict(set)
for a, b, c in high_corr_pairs:
    graph[a].add(b)
    graph[b].add(a)

visited = set()
groups = []

def dfs(node, group):
    visited.add(node)
    group.append(node)
    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs(neighbor, group)

for node in graph:
    if node not in visited:
        group = []
        dfs(node, group)
        groups.append(group)

print(f"Found {len(groups)} correlated groups")
for g in groups:
    print(g)

Found 18 correlated groups
['HLTHPLN1', '_HCVU651']
['BPHIGH4', '_RFHYPE5']
['BLOODCHO', '_CHOLCHK']
['TOLDHI2', '_RFCHOL']
['ASTHMA3', '_LTASTH1', '_ASTHMS1', '_CASTHM1']
['HAVARTH3', '_DRDXAR1']
['EDUCA', '_EDUCAG']
['EXERANY2', '_TOTINDA']
['EXERHMM1', 'PADUR1_']
['HIVTST6', '_AIDTST3']
['_PRACE1', '_MRACE1']
['_AGEG5YR', '_AGE_G', 'MAXVO2_', 'FC60_', '_AGE80']
['HTIN4', 'HTM4']
['DROCDY3_', '_DRNKWEK']
['_MISFRTN', '_FRTRESP', '_FRUITEX', '_MISVEGN', '_VEGRESP', '_VEGETEX']
['_MINAC11', 'PAMIN11_']
['_MINAC21', 'PAMIN21_']
['_PACAT1', '_PA150R2', '_PAINDX1', '_PAREC1', '_PA300R2']


In [39]:
name_to_idx = {name: i for i, name in enumerate(feature_names_v3)}
age_fitness_group = ['_AGEG5YR', '_AGE_G', 'MAXVO2_', 'FC60_', '_AGE80']
for name in age_fitness_group:
    print(f"{name:12s} missing_frac={nan_frac_v3[name_to_idx[name]]:.4f}")

_AGEG5YR     missing_frac=0.0000
_AGE_G       missing_frac=0.0000
MAXVO2_      missing_frac=0.0120
FC60_        missing_frac=0.0120
_AGE80       missing_frac=0.0000


In [40]:
# Define which features to drop based on correlation: keep only features with less missing values at the beginning in each correlation group
name_to_idx = {name: i for i, name in enumerate(feature_names_v3)}
forced_keep = {'_AGEG5YR'}

to_drop = set()
for g in groups:
    if any(name in forced_keep for name in g):
        best = next(name for name in g if name in forced_keep)
    else:
        best = min(g, key=lambda name: nan_frac_v3[name_to_idx[name]])
    for name in g:
        if name != best:
            to_drop.add(name)

print(f"Dropping {len(to_drop)} redundant columns")
print(sorted(to_drop))

Dropping 30 redundant columns
['FC60_', 'HTIN4', 'MAXVO2_', 'PADUR1_', 'PAMIN11_', 'PAMIN21_', '_AGE80', '_AGE_G', '_AIDTST3', '_ASTHMS1', '_CASTHM1', '_CHOLCHK', '_DRDXAR1', '_DRNKWEK', '_EDUCAG', '_FRTRESP', '_FRUITEX', '_HCVU651', '_LTASTH1', '_MISVEGN', '_MRACE1', '_PA300R2', '_PACAT1', '_PAINDX1', '_PAREC1', '_RFCHOL', '_RFHYPE5', '_TOTINDA', '_VEGETEX', '_VEGRESP']


In [41]:
# Drop features
drop_idx = [name_to_idx[name] for name in to_drop]

keep_mask4 = np.ones(x_train_imputed.shape[1], dtype=bool)
keep_mask4[drop_idx] = False

x_train_v4 = x_train_imputed[:, keep_mask4]
x_test_v4 = x_test_imputed[:, keep_mask4]
feature_names_v4 = [f for f, k in zip(feature_names_v3, keep_mask4) if k]

print(x_train_v4.shape)
print(x_test_v4.shape)
print(len(feature_names_v4))

(328135, 117)
(109379, 117)
117


### 2.8 Standardization

In [42]:
mu = np.mean(x_train_v4, axis=0)
sigma = np.std(x_train_v4, axis=0)

# safety check: avoid divide-by-zero for any accidentally-constant column
sigma[sigma == 0] = 1

x_train_final = (x_train_v4 - mu) / sigma
x_test_final = (x_test_v4 - mu) / sigma  # use TRAIN's mu/sigma, not test's own!

print(x_train_final.shape)
print(x_test_final.shape)
print("Mean check (should be ~0):", np.mean(x_train_final, axis=0)[:5])
print("Std check (should be ~1):", np.std(x_train_final, axis=0)[:5])

(328135, 117)
(109379, 117)
Mean check (should be ~0): [-1.46207645e-16 -7.24975111e-17  2.07531704e-16  2.54477524e-16
  1.24207205e-16]
Std check (should be ~1): [1. 1. 1. 1. 1.]


### 2.9 Check class imbalance

In [43]:
print("Fraction y=1 (heart attack):", np.mean(y_train == 1))
print("Fraction y=-1 (no heart attack):", np.mean(y_train == -1))

Fraction y=1 (heart attack): 0.08830207079403295
Fraction y=-1 (no heart attack): 0.911697929205967


Highly imbalanced data set: A trivial model that always predicts "no heart attack" would already get 91.2% accuracy. Accuracy alone is a misleading metric here. Look at metrics like F1-score, precision, recall, or balanced accuracy to know if model is actually learning to detect the minority class (heart attack cases), not just exploiting the imbalance.

## 3. Train/validation split

In [44]:
np.random.seed(42)  # for reproducibility
n = x_train_final.shape[0]
indices = np.random.permutation(n)

val_frac = 0.2
val_size = int(n * val_frac)
val_idx, train_idx = indices[:val_size], indices[val_size:]

x_tr, y_tr = x_train_final[train_idx], y_train[train_idx]
x_val, y_val = x_train_final[val_idx], y_train[val_idx]

print("Train:", x_tr.shape, "Val:", x_val.shape)

Train: (262508, 117) Val: (65627, 117)


# Save cleaned arrays to disk to not have to re-run whole pipeline every time. Remove before submitting !!!!

In [45]:
np.save("x_train_final.npy", x_train_final)
np.save("x_test_final.npy", x_test_final)
np.save("y_train.npy", y_train)
np.save("feature_names_v4.npy", np.array(feature_names_v4))

In [ ]:
import numpy as np

x_train_final = np.load("x_train_final.npy")
x_test_final = np.load("x_test_final.npy")
y_train = np.load("y_train.npy")
feature_names_v4 = np.load("feature_names_v4.npy", allow_pickle=True)  # array of strings needs this flag sometimes